In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

metadata_df = metadata_df[metadata_df['sample_title'].str.contains(r'\[RNA-seq\]', na=False)]
metadata_df = metadata_df.drop(columns=['genotype', 'tumor type', 'batch run'])
metadata_df['sample_title'] = metadata_df['sample_title'].str.replace(' [RNA-seq]', '').str.strip()

metadata_df

,sample_title,tissue,tumor stage,mycn status
samples,,,,
GSM5506066,JB_1,primary neuroblastoma tumor,1,not amplified
GSM5506067,JB_2,primary neuroblastoma tumor,1,not amplified
GSM5506068,JB_3,primary neuroblastoma tumor,1,not amplified
GSM5506069,JB_4,primary neuroblastoma tumor,1,not amplified
GSM5506070,JB_5,primary neuroblastoma tumor,1,not amplified
...,...,...,...,...
GSM5506158,JB_96,primary neuroblastoma tumor,4S,not amplified
GSM5506159,JB_97,primary neuroblastoma tumor,4S,not amplified
GSM5506160,JB_98,primary neuroblastoma tumor,4S,not amplified


In [2]:
SUBSERIES_GEO_ID = 'GSE181559'
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={SUBSERIES_GEO_ID}&format=file&file={SUBSERIES_GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_226/4175934985.py:11: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,tumor stage,mycn status,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM5506066,JB_1,primary neuroblastoma tumor,1,not amplified,1,5,0,0,0,1,...,8591,12,11,11,5290,2123,80,3352,6,26
GSM5506067,JB_2,primary neuroblastoma tumor,1,not amplified,0,1,0,0,0,0,...,7425,3,5,8,4560,5848,8,3443,4,31
GSM5506069,JB_4,primary neuroblastoma tumor,1,not amplified,1,12,0,0,0,0,...,8427,10,27,30,5737,2743,155,2974,8,25
GSM5506070,JB_5,primary neuroblastoma tumor,1,not amplified,0,9,1,0,0,0,...,4234,4,2,2,2828,1164,66,2149,1,7
GSM5506071,JB_6,primary neuroblastoma tumor,1,not amplified,0,2,0,0,0,0,...,15514,59,57,49,6495,2905,152,6765,38,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM5506158,JB_96,primary neuroblastoma tumor,4S,not amplified,3,230,3,2,1,6,...,4564,24,25,37,3320,573,46,2846,67,162
GSM5506159,JB_97,primary neuroblastoma tumor,4S,not amplified,5,238,3,2,0,2,...,11353,47,48,59,5751,1101,97,4717,42,87
GSM5506160,JB_98,primary neuroblastoma tumor,4S,not amplified,15,213,3,1,0,1,...,19341,75,97,133,9418,2507,175,6252,40,118


In [3]:
# Download supplementary data from: https://pmc.ncbi.nlm.nih.gov/articles/instance/10226260/bin/12943_2023_1792_MOESM22_ESM.xlsx

df_clinical_data = pd.read_excel('12943_2023_1792_MOESM22_ESM.xlsx', sheet_name=0)

df_clinical_data['ID'] = df_clinical_data['ID'].astype(str).str.strip()
df_clinical_data['sample_title'] = 'JB_' + df_clinical_data['ID']

df_clinical_data.to_csv('12943_2023_1792_MOESM22_ESM.csv', index=False)

df_combined = df_clinical_data.merge(
    df_combined,
    left_on='sample_title',
    right_on='sample_title',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)
df_combined


,ID,age at diagnosis,dead or last follow up (days),dead,clinical stage,MYCN amplification,17q unbalance,patient number,probe ID tumor,RNA-seq data,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,1,356,2121,-,1,-,+,NaN,984760.0,+,...,8591,12,11,11,5290,2123,80,3352,6,26
1,2,1197,1456,-,1,-,-,NaN,1178899.0,+,...,7425,3,5,8,4560,5848,8,3443,4,31
2,4,298,590,-,1,-,-,NaN,2217756.0,+,...,8427,10,27,30,5737,2743,155,2974,8,25
3,5,1852,1631,-,1,-,-,NaN,2238721.0,+,...,4234,4,2,2,2828,1164,66,2149,1,7
4,6,95,54,-,1,-,-,NaN,2238819.0,+,...,15514,59,57,49,6495,2905,152,6765,38,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,96,46,1395,-,4S,-,-,NaN,2858444.0,+,...,4564,24,25,37,3320,573,46,2846,67,162
92,97,3,4408,-,4S,-,-,NaN,2858297.0,+,...,11353,47,48,59,5751,1101,97,4717,42,87
93,98,84,1919,-,4S,-,-,NaN,2859788.0,+,...,19341,75,97,133,9418,2507,175,6252,40,118
94,99,73,3478,-,4S,-,-,NaN,2858357.0,+,...,2300,13,13,22,1482,703,51,1031,29,48


# Generate description

In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, '12943_2023_1792_MOESM22_ESM.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
